In [ ]:
import numpy as np
import torch
import torch.optim as optim
import matplotlib.pyplot as plt

from ltl.constraint import Constraint
from plane_navigation import *
from util import *

from datetime import datetime
import time

# Case studies with trajectories represented as sets of points (and maybe velocities)

The next cell contains several settings, including:

1. for all case studies:
    * `exectype`: Controls which executable (generated code) is used for the loss.
        Valid options are `ocaml`, `haskell`, `ocaml_torch`, but it may be that only `ocaml` executables are provided.
    * `dt`: The assumed time difference between consecutive states/samples of the
        trajectory. Affects values of velocities, accelerations, etc.
    * `N`: Number of samples on the trajectory.
    * `d`: Dimension of the space. Default to 2, change at your own risk.
    * `init_line`: Parameters to control the line used as an initial state.
        * `start`, `stop`: The starting and ending points, respectively.
        * `rand`: Size of the random component added.

2. for double-loop case studies only:
    * `use_imitation_loss`: If True, add an imitation loss to encourage a trajectory
        close to the first diagonal.

There are also settings for torch and matplotlib.
Some additional settings are controlled at the start of each case study.

If the loss executables return `NaN`s, a heuristic that has worked anecdotally
is that smaller learning rates and larger smoothing parameter (still $<1$)
reduce this problem, particularly as constraints get more complicated.

In [ ]:
exectype = "ocaml"
dt = 1
N = 50
d = 2
init_line = {
    "start": (0.,)*d, "stop": (1.,)*d, "num": N,
    "dt": dt, "rand": 0.005, "come_back": False
}

# Double loop settings
# use_imitation_loss = False  # set weight instead, at experiment

# torch settings
USE_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if USE_CUDA else "cpu")

# Matplotlib settings
plt.rcParams.update({'font.size': 12})

In [ ]:
# Pick a seed, any seed
seed = 17736257
# seed = random.randrange(sys.maxsize)

G = np.random.default_rng(seed)
torch.manual_seed(seed)
np.random.seed(seed)

# I'm not sure about all the things this next setting affects, especially on CUDA.
# Feel free to comment it out (impact on reproduceability?).
# https://pytorch.org/docs/stable/generated/torch.use_deterministic_algorithms.html#torch.use_deterministic_algorithms
# https://pytorch.org/docs/stable/notes/randomness.html
torch.use_deterministic_algorithms(True)

## Avoiding a circle

In [ ]:
max_iter = 600         # maximal number of training iterations
lr = 0.1                # learning rate
smoothing_parameter=0.05

# point p to avoid (centre of the circle)
avoid_pt = torch.Tensor([0.4,0.4]).to(DEVICE)

avoid_radius = 0.1
speed_limit = 0.15 / dt
acceleration_limit = 0.02 / (dt**2)

In [ ]:
constraint = Constraint(
    (                       # constraint string, indexes into post-transform state: 
        "G ( (3 <= 0) & " + # distance to point always greater than const
            "(1 <= 4) & " + # total speed always less than const
            "(2 <= 5) )"    # total acceleration always less than const
    ),
    lambda x : torch.cat(
        [   # first part of post-transform state:
            x.distance_from(avoid_pt, in_constraint=True),
            x.total_speed(True),
            x.total_acceleration(True)
        ],
        dim=-1
    ),
    [                               # second part of post-transform state:
        avoid_radius,               # 3: distance threshold to point
        speed_limit,                # 4: speed limit
        acceleration_limit,         # 5: acceleration limit
    ],
    smoothing_parameter,
    exectype
)

In [ ]:
Z = FixedEndpointTrajectory.new_line(**init_line, rand_gen=G)
Zs = [Z.detach_clone()]
losses = [constraint.loss(Z).detach().numpy()]

In [ ]:
optimizer = optim.SGD(list(Z.parameters()), lr=lr)

In [ ]:
def train(Z):
    # get LTL loss (trajectory as: batch x time x state)
    loss = constraint.loss(Z)

    # backward pass and update
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss

In [ ]:
for i in range(max_iter):
    loss = train(Z)
    losses.append(loss.detach().numpy())
    Zs.append(Z.detach_clone())
    if torch.any(torch.isnan(Z.points)):
        break_msg = ""
        if torch.all(torch.isnan(Z.points)):
            break_msg = " ALL ELEMENTS"
        else:
            break_msg = "\n%s"%(torch.isnan(Z.points))
        print("Z has NaN elements at:" + break_msg)
        break
    if loss <= 5e-3:
        print("Stopped at iteration %i"%(i))
        break

In [ ]:
fig, (ax_pos, ax_vel) = plt.subplots(
    nrows=2, sharex=True,
    gridspec_kw={'height_ratios': [6, 1], 'hspace': 0},
)

# plot the point we're avoiding
ax_pos.scatter(avoid_pt[0], avoid_pt[1], c='r')
avoid_circle = plt.Circle(avoid_pt, radius=avoid_radius, edgecolor='r', fill=False, linestyle="dotted")
avoid_circle.set_label("Avoid")
ax_pos.add_artist(avoid_circle)

# plot initial trajectory
ixy = Zs[0].get_position().detach()
ax_pos.plot(ixy[:,0], ixy[:,1], label="Initial", linestyle='dashed', c="black")

# plot final (learnt/constrained) trajectory
xys = Zs[-1].get_position().detach()
ax_pos.plot(xys[:,0], xys[:,1], label="Final", c="orange", linewidth=1)
ax_pos.scatter(xys[:,0], xys[:,1], c="orange")

# trajectory subplot settings
ax_pos.set_ylabel("y-coordinate ", loc="top")
ax_pos.axis("equal")
ax_pos.set_ylim([0.0-(1.0/7),1.0])
ax_pos.legend(loc="upper left")

# plot velocity
us = Zs[-1].get_speed().detach()
ax_vel.plot(xys[1:,0], us[1:,0], label="x-speed", c='black', linestyle="dashed")
ax_vel.plot(xys[1:,0], us[1:,1], label="y-speed", c='g')

# velocity subplot settings
ax_vel.legend(loc="upper left")
ax_vel.yaxis.set_label_position("right")
ax_vel.yaxis.tick_right()
ax_vel.set_ylabel(" velocity of final path", loc="bottom")

plt.xlabel("x-coordinate")
plt.tight_layout()
plt.show()
# plt.savefig("direct-2-a")

In [ ]:
plt.plot(losses, label=["loss (ltl)"])
plt.legend()
plt.show()

## LTL to smoothen a random but dynamically valid trajectory

The choice for the LTL constraint is not too important. Anything that
is sufficiently encouraging of smooth or ``nice'' trajectories will work
as a demonstration.

We use a very small acceleration limit, to encourage as constant a speed as possible.

In [ ]:
max_iter = 100          # maximal number of training iterations (without LTL)
max_iter_ltl = 200      # maximal number of training iterations (with LTL constraint)
lr = 0.1                # learning rate
smoothing_parameter = 0.01
weight_ltl = 5

speed_limit = 0.03 / dt
acceleration_limit = 0.001 / (dt**2)

endpoints = (
    torch.Tensor(init_line["start"]),
    torch.Tensor(init_line["stop"])
)

We learn once using only the dynamical loss (which leads to a trajectory with valid
velocities but erratic movement), and once using the same random initial state,
but adding an LTL constraint to the dynamical loss term. This constraint
is meant to encourage a more reasonable behaviour.

*Numerical problems may arise for small values in the atomic comparisons, rescale values accordingly.*

Some alternatives for constraints are given below. The paper shows results using
```
G ( (0 >= 2) & (1 >= 2) & (0 <= 3) & (1 <= 3) )
```
which constrains the velocity to be positive (pointing "top-right") and componentwise bounded above.

In [ ]:
# constraint = Constraint(
#     "G ( (0 <= 1) & (4 < 5) & (2 >= 6) & (3 >= 6) )",
#     lambda x: torch.cat([
#             x.points,                   # keep position
#             x.get_speed(prepend=True),  # and velocity
#             L2(x.get_acceleration(prepend=True) * (dt**2)).unsqueeze(-1)],   # and append total acceleration
#         dim=-1).unsqueeze(0),
#     [acceleration_limit * (dt**2), 0.0], # acceleration_limit * (dt**2)
#     smoothing_parameter,
#     exectype
# )

In [ ]:

c0 = Constraint(    # position constraint
    "G ( (0 <= 1) )",
    lambda x: (x.points).unsqueeze(0),
    [],
    smoothing_parameter,
    exectype
)
c1a = Constraint(    # velocity constraint (component-wise min)
    "G ( (0 >= 2) & (1 >= 2) )",
    lambda x: (x.get_speed()).unsqueeze(0),
    [0.0],
    smoothing_parameter,
    exectype
)
c1b = Constraint(    # speed constraint (max)
    "G ( (0 <= 1) )",
    lambda x: L2(x.get_speed()).unsqueeze(-1).unsqueeze(0),
    [speed_limit],
    smoothing_parameter,
    exectype
)

constraint = Constraint(    # upper and lower speed limits
    "G ( (0 >= 2) & (1 >= 2) & (0 <= 3) & (1 <= 3) )",
    lambda x: x.get_speed().unsqueeze(0),
    [0.0, speed_limit],
    smoothing_parameter,
    exectype
)

constraints = [constraint]

In [ ]:
Z = PhaseSpaceTrajectory.new_random(N, d, dt, low=endpoints[0][0], high=endpoints[1][0], rand_gen=G)

Zs = [Z.detach_clone()]
Zs_ltl = [Z.detach_clone()]
losses = []
losses_ltl= []

In [ ]:
optimizer = optim.SGD(list(Z.parameters()), lr=lr)

In [ ]:
def train(Z, weight_ltl=None):

    dyn_loss = Z.get_dynamic_loss(fix_endpoints=endpoints)

    if weight_ltl is not None:
        # get LTL loss (trajectory as: batch x time x state)
        ltl_losses = [c.loss(Z) for c in constraints]
        ltl_loss = sum(ltl_losses)
        loss = dyn_loss + weight_ltl * ltl_loss
        out_loss = [dyn_loss.detach()]
        out_loss.extend([l.detach() for l in ltl_losses])
    else:
        loss = dyn_loss
        out_loss = dyn_loss.detach()

    # backward pass and update
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return out_loss

Train with dynamical loss only at first.

In [ ]:
for i in range(max_iter):
    loss = train(Z)
    losses.append(loss)
    Zs.append(Z.detach_clone())
    if torch.any(torch.isnan(Z.points)):
        if torch.all(torch.isnan(Z.points)):
            break_msg = " ALL ELEMENTS"
        else:
            break_msg = "\n%s"%(torch.isnan(Z.points))
        print("Z has NaN elements at:" + break_msg)
        break
    if (loss.detach().numpy() <= 1e-10):
        print("Stopped at iteration %i"%(i))
        break
print(f"Final loss: {loss.detach().item()}")

fig, ax_x, ax_v = plot_trajectory_and_speed(Zs[0], Zs[-1], ylim=[-1/7,1])
fig.tight_layout()

Then train with the LTL added, and plot a comparison.

In [ ]:
Z = Zs_ltl[0].detach_clone()
optimizer = optim.SGD(list(Z.parameters()), lr=lr)

In [ ]:
for i in range(max_iter_ltl):
    ls = train(Z, weight_ltl=weight_ltl)
    dyn = ls[0]
    ltls = ls[1:]
    losses_ltl.append(ls)
    loss = dyn + weight_ltl*sum(ltls)
    Zs_ltl.append(Z.detach_clone())
    if torch.any(torch.isnan(Z.points)):
        if torch.all(torch.isnan(Z.points)):
            break_msg = " ALL ELEMENTS"
        else:
            break_msg = "\n%s"%(torch.isnan(Z.points))
        print("Z has NaN elements at:" + break_msg)
        assert False
    if (loss.detach().numpy() <= 3e-2):
        print("Stopped at iteration %i"%(i))
        break

fig, ax_x, ax_v = plot_trajectory_and_speed(Zs[-1], Zs_ltl[-1], labels=["Dynamical only", "Dynamical and LTL"], ylim=[-1/7,1])
fig.tight_layout()

In [ ]:
plt.plot([[l[0], weight_ltl * l[1]] for l in losses_ltl], label=["dynamical loss", "LTL loss (weighted)"])
# plt.plot(losses_ltl, label=["dynamical loss", "LTL (position)", "LTL (speed min)", "LTL (speed max)"])
plt.ylim([-0.1,2])
plt.legend()
plt.show()

## Double loop case studies: Conjoined and nested constraints

We attempt to obtain a trajectory following two consecutive loops in different directions by specifying some points it should pass through in order. Of course this constraint is not fail-safe: individual points don't specify continuous shapes. It turns out that a constraint with an ordered quadruplet of points is needed for the case studies in this notebook (i.e. naive point-set trajectories), whereas two ordered pairs suffice for the DMPs in the other notebook.

### Conjoined constraint

The constraint here will be
$$\Diamond (\|p-o_2\|\leq0 \wedge \mathcal{X} (\Diamond (\|p-o_1\|\leq0))) \wedge \Diamond (\|p-o_4\|\leq0 \wedge \mathcal{X} (\Diamond (\|p-o_3\|\leq0)))$$
for points $o_1$ as in the graphs below. In code, `o_i = reach_points[i]`.

Notice also the possibility to add imitation loss, for comparison with the corresponding DMP case study. The imitation loss in this case is just the difference to the first diagonal (either a pointwise diagonal, regularly sampled, or the sum of normal distances to the diagonal line).

In [ ]:
max_iter = 1000              # maximal number of training iterations (without LTL)
lr = 0.05                   # learning rate
smoothing_parameter = 0.05
dynamics_weight = 0.1       # weight for the dynamical loss functions
imitation_weight = 0        # For the imitation loss. Set to 0 to turn off.

speed_limit = 0.05 / dt
# acceleration_limit = 0.15 / (dt**2)
until_radius = 0.0          # maximal desired distance to the points the trajectory should reach

init_default = {
    "start": (0.,)*d, "stop": (1.,)*d, "num": N,
    "dt": dt, "rand": 0.0025, "come_back": False
}

In [ ]:
reach_points = [
    torch.Tensor(p).to(DEVICE)
    for p in [[0.2,0.6], [0.4,0.8], [0.6,0.2], [0.8,0.4]]
]
reach_Points = [Point.from_tensor(rp) for rp in reach_points]

Define 2 ways to calculate an imitation loss: as a distance from a continuous line, or as pairwise distances from regular samples on a line (i.e. a list of points like our representation of trajectories). The latter, i.e. `imitation_loss_points` is more similar to the DMP case study. Be mindful that this implies velocity is kept to the constant-velocity-diagonal demonstration as well as position.

In [ ]:
conjoined_loop = PhaseSpaceTrajectory.double_loop_conjoined(
    speed_limit=speed_limit,
    points=reach_Points,
    reach_margin=until_radius,
    smoothing_parameter=smoothing_parameter
)
print(conjoined_loop.ltl)

def imitation_loss(
    Z: torch.Tensor,
    line=(torch.Tensor(init_default["start"]),torch.Tensor(init_default["stop"])),
    sum=True):
    """
    Args:
        Z: shape (N,D) where D is the dimension of each point on the trajectory
        line: pair of points (`torch.Tensor`s) defining the desired line
        sum: by default (True) return the sum of the computed distances;
            else (False), return distances as a `tensor`

    Returns:
        The (`tensor` or sum of) perpendicular distances to a given line.
    """
    a, b = line
    a.to(Z.dtype)
    b.to(Z.dtype)
    # pytorch only has 3D cross product. Could pad and use third dim...
    # https://discuss.pytorch.org/t/best-way-to-do-a-2d-cross-product/106522/2
    # ... but that seems wasteful (lots of discarded computations)
    cross2D = ((b[0]-a[0])*(Z[:,1]-a[1])) - ((b[1]-a[1])*(Z[:,0]-a[0]))
    # Obtain the area of the parallelogram spanned by `line` and a point of `Z`
    area = torch.abs(cross2D)
    # Divide by the length of the base (b-a) to obtain the height
    # of the parallelogram (i.e. normal distance to the point on Z)
    height = area/L2(b-a)
    if sum:
        return torch.sum(height)
    return height

def imitation_loss_points(Z: torch.Tensor, line: torch.Tensor):
    """
    Returns the (sum of) perpendicular distances to a given line.

    Args:
        Z: shape (N,D) where D is the dimension of each point on the trajectory
        line: pair of points (`torch.Tensor`s) defining the desired line
    """
    assert Z.shape == line.shape
    return L2(Z-line)

Select which imitation loss (`imi_loss`) you want by (un)commenting the corresponding lines.

In [ ]:
def train(Z, dynamics=dynamics_weight, imitation_weight=imitation_weight):
    # get LTL loss
    loop_loss = conjoined_loop.loss(Z)
    dyn_loss = Z.get_dynamic_loss(fix_endpoints=(p1,p2))

    # imitation loss
    imi_loss = imitation_loss(Z.points)
    # imi_loss = imitation_loss_points(Z.points, PhaseSpaceTrajectory.new_line(
    #         init_default["start"], init_default["stop"], init_default["num"], init_default["dt"]
    # ))

    loss = loop_loss + dynamics*dyn_loss + imitation_weight*imi_loss

    # backward pass and update
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    return loss.item(), loop_loss.item(), dyn_loss.item(), imi_loss.item()

<!-- Use several seeds to get a set of examples from different initial states -->

In [ ]:
# seeds = [0,1,17,2238]
# results_by_seed = {}

In [ ]:
Z = PhaseSpaceTrajectory.new_line(**init_default, rand_gen=None)

z = Z.detach_clone()
Zs = [z]
p1 = torch.Tensor(init_default["start"])
p2 = torch.Tensor(init_default["stop"])
with torch.no_grad():
    ltl = conjoined_loop.loss(Z).item()
    dyn = Z.get_dynamic_loss(fix_endpoints=(p1,p2)).item()
    imi = imitation_loss(Z.points)
losses = [(ltl + dynamics_weight*dyn + imi, ltl, dyn, imi)]

In [ ]:
optimizer = optim.SGD(list(Z.parameters()), lr=lr)

In [ ]:
t0 = time.time()
for i in range(max_iter):
    loss, loop, dyn, imi = train(Z, dynamics=dynamics_weight)
    losses.append([loss, loop, dyn, imi])
    z = Z.detach_clone()
    Zs.append(z)
    if i%100 == 0:
        runtime(t0)
        print("iteration %i, total loss %f"%(i, loss))
        print(f"{loop:.5} | {dyn:.5} | {imi:.5}")

In [ ]:
fig, ax = plt.subplots()
double_loop_plot(ax, Zs, reach_points, arrow_threshold=0.09)
plt.tight_layout()
plt.show()

In [ ]:
if imitation_weight:
    plt.plot(losses, label=["total", "loop (LTL)", "dynamic", "imitation"])
else:
    plt.plot([[t,l,d] for t,l,d,i in losses], label=["total", "loop (LTL)", "dynamic"])
plt.legend()
plt.show()

### Nested Constraint

The constraint for this case study is
$$\Diamond (\|p-o_2\|\leq0 \wedge \mathcal{X} (\Diamond (\|p-o_1\|\leq0) \wedge \mathcal{X} (\Diamond (\|p-o_4\|\leq0) \wedge \mathcal{X} (\Diamond (\|p-o_3\|\leq0)))))$$
i.e. the same as the last case study, but with the conjunction of two paris of points replaced with a single ordered quadruplet. Slower to run due to the nested temporal operators: anecdotally, over 70 times slower.

In [ ]:
max_iter = 80
smoothing_parameter = 0.01
lr = 0.05                           # learning rate
dynamics_weight = 0.1               # weight for the dynamical loss functions

speed_limit = 0.05 / dt
until_radius = 0.0

init_default = {
    "start": (0.,)*d, "stop": (1.,)*d, "num": N,
    "dt": dt, "rand": 0.0025, "come_back": False
}

In [ ]:
reach_points = [
    torch.Tensor(p).to(DEVICE)
    for p in [[0.2,0.6], [0.4,0.8], [0.6,0.2], [0.8,0.4]]
]
reach_Points = [Point.from_tensor(rp) for rp in reach_points]

In [ ]:
nested_loop = PhaseSpaceTrajectory.double_loop_nested(
    speed_limit=speed_limit,
    points=reach_Points,
    reach_margin=until_radius,
    smoothing_parameter=smoothing_parameter
)
print(nested_loop.ltl)

In [ ]:
Z = PhaseSpaceTrajectory.new_line(**init_default, rand_gen=G)

z = Z.detach_clone()
Zs = [z]
p1 = torch.Tensor(init_default["start"])
p2 = torch.Tensor(init_default["stop"])
with torch.no_grad():
    ltl = nested_loop.loss(Z).item()
    dyn = Z.get_dynamic_loss(fix_endpoints=(p1,p2)).item()
losses = [(ltl + dynamics_weight*dyn, ltl, dyn)]

In [ ]:
def train(Z, dynamics=dynamics_weight):
    # get LTL loss
    loop_loss = nested_loop.loss(Z)
    dyn_loss = Z.get_dynamic_loss(fix_endpoints=(p1,p2))
    loss = loop_loss + dynamics*dyn_loss

    # backward pass and update
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item(), loop_loss.item(), dyn_loss.item()

In [ ]:
optimizer = optim.SGD(list(Z.parameters()), lr=lr)

t0 = time.time()
for i in range(max_iter):
    loss, loop, dyn = train(Z, dynamics=dynamics_weight)
    losses.append((loss, loop, dyn))
    z = Z.detach_clone()
    Zs.append(z)
    if i%5 == 0:
        runtime(t0)
        print("iteration %i, loop %f, dyn %f"%(i, loop, dyn))

In [ ]:
fig, ax = plt.subplots()
double_loop_plot(ax, Zs, reach_points, arrow_threshold=0.15)
plt.tight_layout()
plt.show()

In [ ]:
# plt.plot([[l,d] for t,l,d in losses], label=["loop (LTL)", "dynamics"])
plt.plot(losses, label=["total", "loop (LTL)", "dynamics"])
plt.legend()
plt.show()